# Rongali2018 — Emissividade

Estima emissividades distintas para B10 e B11 por mistura linear, usando os extremos NDVI 0,15/0,48 da cena estudada por Rongali. Esses extremos não são universais.

## Precisa de

**Bandas ópticas — obrigatórias na forma de uso deste exemplo:**

- **Vermelho:** imagem calibrada em radiância TOA, em **W m⁻² sr⁻¹ µm⁻¹**.
- **Infravermelho próximo (NIR):** imagem calibrada na **mesma grandeza e unidade do vermelho**.

A função calcula o NDVI internamente. Como alternativa, fornecer somente `ndvi_image` já preparado, **adimensional, em [-1,1]**, no lugar das duas bandas; não combinar as entradas.

**Banda de destino:**

- **`band`:** número da banda, **sem unidade**; usar **10 ou 11**, selecionando os parâmetros da banda.

O exemplo segue a interpretação de radiância TOA do Step 1 de Rongali; transferir os extremos NDVI para outra cena requer justificativa.

**Entrada opcional de qualidade:**

- **`mask`:** imagem **booleana**, sem unidade: `True` exclui o pixel; `False` permite avaliar, sujeito às validações do método. Neste exemplo, sua origem é descrita na seção de leitura.

Todas as imagens usadas juntas devem ter a mesma quantidade de linhas/colunas, posição dos pixels, resolução e sistema de coordenadas.

## Arquivos para executar este notebook

O exemplo usa o leitor de cena **Landsat 8/9 Collection 2 Level 1**. Coloque na pasta da cena:

- **B4:** arquivo original do vermelho, em **DN**; o leitor converte para radiância TOA, em W m⁻² sr⁻¹ µm⁻¹.
- **B5:** arquivo original do NIR, em **DN**; o leitor converte para radiância TOA, em W m⁻² sr⁻¹ µm⁻¹.
- **B10:** arquivo térmico original, em **DN**; o leitor converte para BT10 em **K**.
- **B11:** arquivo térmico original, em **DN**; o leitor converte para BT11 em **K**.
- **MTL.txt:** metadados e coeficientes de calibração da própria cena; não é uma imagem.
- **QA_PIXEL:** códigos inteiros de qualidade, sem unidade física; o leitor interpreta os bits para montar a máscara.
- **QA_RADSAT:** códigos inteiros de saturação e oclusão de terreno, sem unidade física; complementam a máscara.

Essa lista corresponde ao leitor usado no notebook; uma chamada direta à função exige somente as entradas indicadas em “Precisa de”. B2 e SZA são opcionais para o leitor. Preserve os nomes originais e a grade dos arquivos. Não passar os arquivos QA brutos como máscara booleana.

## Como as imagens são preparadas

**TOA significa topo da atmosfera.** A grandeza também precisa ser especificada:

- **Radiância TOA:** energia medida pelo sensor, em **W m⁻² sr⁻¹ µm⁻¹**.
- **Reflectância TOA:** **adimensional**, com ajuste do ângulo solar.
- **Temperatura de brilho TOA:** temperatura equivalente da radiância térmica, em **K**.

O leitor usa o **MTL da própria cena** para calibrar os arquivos originais. Não é necessário preparar manualmente as imagens TOA para executar este notebook.

A configuração `optical_quantity="radiance"` devolve as bandas ópticas em **radiância TOA, em W m⁻² sr⁻¹ µm⁻¹**; as bandas térmicas são devolvidas como temperaturas de brilho em **K**.

Trocar para `optical_quantity="reflectance"` muda a calibração óptica; isso exige verificar a compatibilidade com a parametrização usada no método. **Reflectância TOA ainda contém efeitos atmosféricos e é diferente de reflectância de superfície.**

Edite os caminhos na célula de configuração e execute de cima para baixo. As saídas ficam em `resultados/`; os arquivos originais não são modificados.


## Função geral — parâmetros aplicáveis ao método

Argumentos nomeados e relacionados ficam próximos. A tabela abaixo segue **exatamente a ordem desta chamada**. Obrigatório: necessário ao cálculo; opcional: pode usar o padrão; condicional: depende da rota escolhida. Valores de configuração mostrados podem diferir do padrão da API.

```python
resultado = emissivity(
    # Método
    method='gopinadh-2018',

    # Banda de destino
    band=10,

    # Entradas para o NDVI
    red=opticas[4],
    nir=opticas[5],

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `None → "avdan-2016"` | Seletor | Usar "gopinadh-2018" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `band` | Opcional; recomendado identificar | `None → 10` | Número da banda | 10 ou 11; aliases b10/b11 e band_10/band_11 aceitos. Fornece os parâmetros da banda selecionada. |
| `red` | Obrigatório nesta rota; omitir com NDVI preparado | `None` | W m⁻² sr⁻¹ µm⁻¹ neste exemplo | Banda vermelha calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `nir` | Obrigatório nesta rota; omitir com NDVI preparado | `None` | W m⁻² sr⁻¹ µm⁻¹ neste exemplo | Banda NIR calibrada e alinhada. Mesma grandeza radiométrica da outra banda; não misturar DN, radiância e reflectância. NDVI calculado internamente. Respeitar a convenção óptica indicada na introdução. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |

## Alternativa — NDVI preparado

Usar NDVI já calculado e alinhado no lugar do par vermelho/NIR. Não há necessidade de chamar uma função pública de NDVI.

```python
resultado = emissivity(
    # Método
    method='gopinadh-2018',

    # Banda de destino
    band=10,

    # Entradas para o NDVI
    ndvi_image=ndvi_preparado,

    # Exclusão de pixels
    mask=mascara,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `None → "avdan-2016"` | Seletor | Usar "gopinadh-2018" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `band` | Opcional; recomendado identificar | `None → 10` | Número da banda | 10 ou 11; aliases b10/b11 e band_10/band_11 aceitos. Fornece os parâmetros da banda selecionada. |
| `ndvi_image` | Condicional: alternativa ao par vermelho/NIR | `None` | Adimensional, [-1,1] | NDVI já preparado e alinhado; omitir as duas bandas ópticas. A convenção radiométrica deve respeitar a fonte do método. Não combinar NDVI preparado e vermelho/NIR. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |


In [ ]:
from pathlib import Path
import numpy as np
from pylstemp import emissivity
from pylstemp.scene import LandsatScene

pasta = Path("CAMINHO_DA_CENA")
saida = Path("resultados/gopinadh_2018")
saida.mkdir(parents=True, exist_ok=True)
banda = 10

## Ler e calibrar a imagem

`LandsatScene` confere os arquivos/grade da cena e le o MTL. `calibrated()` retorna bandas opticas, BT10/BT11 em K e mascara QA booleana. As entradas deste exemplo sao radiancias opticas TOA.

### Máscara usada neste exemplo

`mask` é uma máscara **de exclusão**, não uma máscara de seleção: `True` rejeita o pixel e `False` permite o cálculo. O leitor usa QA_PIXEL para fill, nuvem dilatada, cirrus de alta confiança, nuvem, sombra e neve; QA_RADSAT para saturação óptica B2/B4/B5 e oclusão de terreno; e verifica SZA quando disponível. A opção `exclude_water=True` também exclui água (bit7 de QA_PIXEL). Valores não finitos ou fora do domínio podem ser rejeitados adicionalmente pelo cálculo. `mask=None` em uma chamada direta não cria automaticamente uma máscara de nuvens.


In [ ]:
cena = LandsatScene(pasta)
opticas, bt, mascara = cena.calibrated(optical_quantity="radiance")

## Aplicar o metodo

Adota-se a interpretacao de radiancia TOA do Step1 de Rongali. Transferir os extremos .15/.48 para outra cena requer justificativa.

In [ ]:
imagem = emissivity(
    # Método
    method='gopinadh-2018',

    # Banda de destino
    band=banda,

    # Entradas para o NDVI
    red=opticas[4],
    nir=opticas[5],

    # Exclusão de pixels
    mask=mascara,
)


## Salvar a imagem

A saida preserva a grade original e e gravada como float32 com NoData NaN. A unidade da emissividade/reflectancia e adimensional; PWV usa g/cm2; BT usa K e LST respeita `unit`.

In [ ]:
cena.write_geotiff(saida / "gopinadh_2018.tif", imagem, units='dimensionless')
cena.close()
print("Pixels validos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saida:", saida)

## Referencia e limites

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).